In [11]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings,ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from youtube_transcript_api._errors import (
    TranscriptsDisabled,
    NoTranscriptFound,
    VideoUnavailable,
    IpBlocked
)
from dotenv import load_dotenv

# Step 1 :- Indexing

In [25]:

video_id = "LPZh9BOjkQs"

api = YouTubeTranscriptApi()

try:
    transcript = api.fetch(video_id, languages=["en"])
    text = " ".join(item["text"] for item in transcript.to_raw_data())
    print(text)

except IpBlocked:
    print("Your IP has been blocked by YouTube.")

except NoTranscriptFound:
    print("No English transcript found.")

except TranscriptsDisabled:
    print("Transcripts are disabled for this video.")

except VideoUnavailable:
    print("Video unavailable.")

except Exception as e:
    print(type(e).__name__, e)

[Submit subtitle corrections at criblate.com]
Imagine you happen across a short movie script that describes a scene between a person and their AI assistant. The script has what the person asks the AI, but the AI's response has been torn off. Suppose you also have this powerful magical machine that can take any text and provide a sensible prediction of what word comes next. You could then finish the script by feeding in what you have to the machine, seeing what it would predict to start the AI's answer, and then repeating this over and over with a growing script completing the dialogue. When you interact with a chatbot, this is exactly what's happening. A large language model is a sophisticated mathematical function that predicts what word comes next for any piece of text. Instead of predicting one word with certainty, though, what it does is assign a probability to all possible next words. To build a chatbot, what you do is lay out some text that describes an interaction between a user

# Step 1b - Indexing (Text Splitting)

In [26]:
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = splitter.create_documents([text])

In [27]:
len(chunks)

95

# Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [28]:
import traceback

try:
    load_dotenv()

    embeddings = GoogleGenerativeAIEmbeddings(
        model="models/gemini-embedding-001"
    )

    vector_store = FAISS.from_documents(
        documents=chunks,
        embedding=embeddings
    )

    print("✅ Vector store created successfully!")

except Exception as e:
    print(f"❌ Error: {e}")
    print("\nFull Traceback:")
    traceback.print_exc()

✅ Vector store created successfully!


In [30]:
vector_store.index_to_docstore_id

{0: '834816ee-d8d6-4606-a36e-9ff6d55b300f',
 1: '0f586f1f-90c6-452f-9e52-82d1a9426f0c',
 2: '726e5515-2bcc-4cdd-9a20-b1dc5bc388d4',
 3: '21daa9e3-9125-4e4b-8c49-e7af67fc07ec',
 4: '320695ea-40fb-4497-8437-ef9a61fd078d',
 5: '9be8d153-d00d-48ed-b5f2-d49215b79ec7',
 6: 'f71f4e77-627a-4c93-b944-06998bd55b90',
 7: 'e96d9279-c00f-4f3e-b4b6-9cdff3985f79',
 8: 'f91b33d6-cc00-454c-8972-e2f99bd0072b',
 9: 'dab3fc17-fbc8-48e3-af6b-e78d1376412c',
 10: '2db52d80-0949-495c-95fb-a0e6c450a0fa',
 11: '4462fb6e-9f95-4fc8-aef5-62e548a1b61f',
 12: 'ae166043-7985-45a6-a6c4-5a0e6930a917',
 13: '398e6ea6-5cf5-46ca-a399-b9b830c93700',
 14: '2b3918c4-0d92-439e-8d0a-8e6a48e2466d',
 15: '506649da-e3ac-46a7-bb24-443cf5778732',
 16: 'ca71dc5d-d476-495c-88ef-c50eabb6adfb',
 17: 'a66c8994-e6ec-40cf-a551-93c625a7649f',
 18: '15639635-f515-4a25-bc5d-222784bdd889',
 19: '21dba79f-46d0-4503-afb9-f5647ad42b5f',
 20: '1a86372a-a1cd-4337-bb3b-6c87cddd40bd',
 21: '713b166a-d597-451e-9381-0fbb8cbc4509',
 22: '0a5d3155-f80b-

In [31]:
vector_store.get_by_ids(['834816ee-d8d6-4606-a36e-9ff6d55b300f'])

[Document(id='834816ee-d8d6-4606-a36e-9ff6d55b300f', metadata={}, page_content='[Submit subtitle corrections at criblate.com]')]

# Step 2 - Retrieval

In [32]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [33]:
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000244124D52B0>, search_kwargs={'k': 4})

In [34]:
retriever.invoke('What is llm')

[Document(id='c7be8ae4-d75b-4c31-85a5-a795dbd80afa', metadata={}, page_content='the large in large language model is how they can have hundreds of billions of these parameters. No'),
 Document(id='1c41721b-07ff-4262-9551-d5097ee93f2b', metadata={}, page_content='when you use large language model predictions to autocomplete a prompt, the words that it generates'),
 Document(id='f91b33d6-cc00-454c-8972-e2f99bd0072b', metadata={}, page_content="a chatbot, this is exactly what's happening. A large language model is a sophisticated mathematical"),
 Document(id='b40bd475-b3fc-483a-ae13-63bd59b01ad3', metadata={}, page_content='of computation involved in training a large language model is mind-boggling. To illustrate, imagine')]

# Step 3 - Augmentation

In [35]:
llm=ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0.2)

In [36]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [42]:
question= "parameter and weight"
retrieved_docs    = retriever.invoke(question)

In [43]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'usually called parameters or weights. Changing those parameters will change the probabilities that\n\nthe large in large language model is how they can have hundreds of billions of these parameters. No\n\nmodel behaves is entirely determined by these many different continuous values, usually called\n\nthe example. An algorithm called backpropagation is used to tweak all of the parameters in such a'

In [44]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [45]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      usually called parameters or weights. Changing those parameters will change the probabilities that\n\nthe large in large language model is how they can have hundreds of billions of these parameters. No\n\nmodel behaves is entirely determined by these many different continuous values, usually called\n\nthe example. An algorithm called backpropagation is used to tweak all of the parameters in such a\n      Question: parameter and weight\n    ")

# Step 4 - Generation

In [46]:
answer = llm.invoke(final_prompt)
print(answer.content)

Parameters and weights are usually called by both terms, indicating they are synonymous. They are many different continuous values that entirely determine how a model behaves. Changing these parameters will change probabilities. Large language models can have hundreds of billions of these parameters, and an algorithm called backpropagation is used to tweak them.


# Building a Chain

In [47]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [48]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [49]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [50]:
parallel_chain.invoke('what is llm')

{'context': "the large in large language model is how they can have hundreds of billions of these parameters. No\n\nwhen you use large language model predictions to autocomplete a prompt, the words that it generates\n\na chatbot, this is exactly what's happening. A large language model is a sophisticated mathematical\n\nof computation involved in training a large language model is mind-boggling. To illustrate, imagine",
 'question': 'what is llm'}

In [51]:
parser = StrOutputParser()

In [52]:
main_chain = parallel_chain | prompt | llm | parser

In [53]:
main_chain.invoke('parameter and weight')

'Parameters and weights are terms often used interchangeably ("usually called parameters or weights"). They are described as "many different continuous values" that entirely determine how a model behaves. Changing these parameters will change probabilities, and large language models can have hundreds of billions of them. An algorithm called backpropagation is used to tweak all of the parameters.'